# Vehicle Service Management & Maintenance Analysis System
## Foundations of Data Science (25ITC31) – Project-Based Assessment
**Academic Year:** 2026-2027 (ODD) | **Class:** 25IT3A&B  
**Scenario 16 (Register Nos: 16, 46, 76, 106)**  

---  
### Project Objectives:
1. **Data Retrieval & Exploration:** Extract vehicle service records, maintenance activities, and billing ledgers.
2. **Data Cleaning & Preprocessing:** Address missing values, convert data types, detect statistical outliers using IQR and Z-scores.
3. **Data Wrangling:** Group by vehicle type, compute aggregations, and construct multidimensional pivot tables.
4. **Exploratory Data Analysis & Visualization:** Build publication-ready charts (Bar, Boxplot, Dual-axis Time Series, Regression, Heatmap, Donut) using Matplotlib and Seaborn.
5. **Actionable Insights:** Formulate maintenance cost optimization and bay workload scheduling recommendations.

In [ ]:
# Unit 2: Importing Essential Data Science Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setting visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})

print("[INFO] Libraries imported successfully.")

In [ ]:
# Unit 2: Data Retrieval from Local Storage / CSV
dataset_path = os.path.join("data", "vehicle_maintenance_dataset.csv")
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape[0]} records, {df.shape[1]} features")
df.head()

In [ ]:
# Data Quality Audit and Schema Inspection
df.info()

In [ ]:
# Unit 4: Data Cleaning & Preprocessing Pipeline
# 1. Convert service_date to datetime
df['service_date'] = pd.to_datetime(df['service_date'])
df['service_month'] = df['service_date'].dt.strftime('%Y-%m')

# 2. Convert categorical columns to category dtypes
for col in ['vehicle_type', 'payment_mode', 'payment_status']:
    df[col] = df[col].astype('category')

# 3. Missing Value Audit
print("Missing values:")
print(df.isnull().sum())

# 4. Feature Engineering
df['labor_parts_ratio'] = np.round(df['labor_charges'] / (df['parts_cost'] + 1.0), 3)
df['effective_hourly_rate'] = np.round(df['labor_charges'] / df['labor_hours'], 2)

print("\nEngineered features added successfully.")
df[['record_id', 'vehicle_type', 'labor_charges', 'parts_cost', 'total_amount', 'labor_parts_ratio']].head()

In [ ]:
# Unit 1: Descriptive Statistics & Standard Z-Scores
mean_cost = df['total_amount'].mean()
std_cost = df['total_amount'].std()
median_cost = df['total_amount'].median()

df['cost_z_score'] = np.round((df['total_amount'] - mean_cost) / std_cost, 3)

print(f"Mean Total Cost   : ₹{mean_cost:.2f}")
print(f"Median Total Cost : ₹{median_cost:.2f}")
print(f"Std Deviation     : ₹{std_cost:.2f}")

# High-cost outlier jobs (Z-score > 1.5)
high_cost_anomalies = df[df['cost_z_score'] > 1.5]
print(f"\nHigh-Cost Maintenance Anomaly Records (Z > 1.5): {len(high_cost_anomalies)}")
high_cost_anomalies[['record_id', 'model', 'vehicle_type', 'service_type', 'total_amount', 'cost_z_score']]

In [ ]:
# Unit 3: Data Wrangling & Multidimensional Pivot Table
pivot_table = pd.pivot_table(
    df,
    values='total_amount',
    index='vehicle_type',
    columns='service_month',
    aggfunc=['count', 'mean'],
    fill_value=0
)
print("PIVOT TABLE: Workload Count & Mean Cost (₹) by Vehicle Type and Month")
pivot_table.round(2)

In [ ]:
# Unit 5 - Visualization 1: Service Type Demand & Booking Frequency (Bar Chart)
plt.figure(figsize=(10, 5))
service_counts = df['service_type'].value_counts()
colors = sns.color_palette("viridis", len(service_counts))
bars = plt.barh(service_counts.index, service_counts.values, color=colors, edgecolor='black', alpha=0.85)
plt.title("Service Type Demand & Appointment Frequency", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Service Appointments", fontsize=12)
plt.ylabel("Service Type", fontsize=12)
for bar in bars:
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f"{int(bar.get_width())}", va='center', ha='left', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Unit 5 - Visualization 2: Maintenance Cost by Vehicle Type (Box Plot)
plt.figure(figsize=(10, 5))
sns.boxplot(x='vehicle_type', y='total_amount', data=df, palette="Set2", showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black", "markersize":"8"})
plt.title("Maintenance Expenditure Distribution by Vehicle Type (₹)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Vehicle Classification", fontsize=12)
plt.ylabel("Total Service Invoice (₹)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Unit 5 - Visualization 3: Monthly Workload & Revenue Trend (Dual-Axis Time Series)
monthly = df.groupby('service_month').agg(
    jobs=('record_id', 'count'),
    revenue=('total_amount', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(monthly['service_month'], monthly['jobs'], color='teal', alpha=0.6, width=0.4, label='Completed Jobs')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Completed Services', color='teal', fontsize=12)
ax1.tick_params(axis='y', labelcolor='teal')

ax2 = ax1.twinx()
ax2.plot(monthly['service_month'], monthly['revenue'], color='crimson', marker='o', linewidth=2.5, label='Revenue (₹)')
ax2.set_ylabel('Total Billed Revenue (₹)', color='crimson', fontsize=12)
ax2.tick_params(axis='y', labelcolor='crimson')

plt.title("Service Center Monthly Workload & Revenue Evolution", fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

In [ ]:
# Unit 5 - Visualization 4: Parts Cost vs. Labor Charges (Scatter Plot with Regression Trendline)
plt.figure(figsize=(10, 5))
sns.regplot(x='parts_cost', y='labor_charges', data=df,
            scatter_kws={'alpha':0.7, 'color':'purple', 's':70},
            line_kws={'color':'darkorange', 'linewidth':2})
plt.title("Correlation: Spare Parts Expenditure vs. Labor Charges (₹)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Total Spare Parts Cost (₹)", fontsize=12)
plt.ylabel("Labor Charges (₹)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Unit 5 - Visualization 5: Multivariate Correlation Heatmap
plt.figure(figsize=(8, 6))
corr = df[['labor_hours', 'labor_charges', 'parts_cost', 'subtotal', 'discount_amount', 'total_amount']].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True, linewidths=1.0)
plt.title("Multivariate Correlation Heatmap (Service Metrics)", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Unit 5 - Visualization 6: Payment Channel Breakdown (Donut Chart)
plt.figure(figsize=(7, 7))
pay_counts = df['payment_mode'].value_counts()
plt.pie(pay_counts.values, labels=pay_counts.index, autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette("pastel"), pctdistance=0.80, explode=[0.05]*len(pay_counts))
centre_circle = plt.Circle((0, 0), 0.60, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title("Customer Payment Channel Preferences", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()